In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

In [2]:
data = pd.read_csv("../data/final_dataset.csv")

In [3]:
df = pd.DataFrame(data)
df.head()

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0


In [4]:
params = [
    "age",
    "income",
    "n_child",
    "sex",
    "type_area",
    "invalid",
    "mar_st",
    "visit_doctor",
    "work",
    "alcohol",
    "smoking",
    "phys_active",
    "is_health_good",
    "is_health_very_good",
    "diploma",
]
targets = df.keys().drop(params).to_list()  # проведем эту операцию еще раз

In [5]:
df[params]

,age,income,n_child,sex,type_area,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,44.0,43000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,61.0,20000.0,3.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,55.5,45000.0,3.0,1.0,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,40.5,50000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,55.0,55000.0,1.0,1.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,50.5,90000.0,4.0,2.0,0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4594,37.5,60000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4595,48.0,20000.0,2.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4596,37.5,20000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df_men = df[df["sex"] == 1]
df_women = df[df["sex"] == 2]

In [ ]:
# def perform_matching(data, treatment_col, outcomes, covars, method="psm"):
#     data_clean = data.dropna(subset=covars + [treatment_col])
#     X = data_clean[covars]
#     T = data_clean[treatment_col]
#     results = {}

#     if method == "psm":
#         lr = LogisticRegression()
#         lr.fit(X, T)
#         data_clean["ps"] = lr.predict_proba(X)[:, 1]
#         treated = data_clean[data_clean[treatment_col] == 1]
#         untreated = data_clean[data_clean[treatment_col] == 0]
#         nn = NearestNeighbors(n_neighbors=1).fit(untreated[["ps"]])
#         _, indices = nn.kneighbors(treated[["ps"]])
#         matched_data = pd.concat([treated, untreated.iloc[indices.flatten()]])
#     else:
#         scaler = StandardScaler()
#         X_scaled = scaler.fit_transform(X)
#         X_scaled_df = pd.DataFrame(X_scaled, index=data_clean.index, columns=covars)
#         treated_scaled = X_scaled_df[data_clean[treatment_col] == 1]
#         untreated_scaled = X_scaled_df[data_clean[treatment_col] == 0]
#         vi = np.linalg.pinv(np.cov(X_scaled.T))
#         nn = NearestNeighbors(
#             n_neighbors=1, metric="mahalanobis", metric_params={"VI": vi}
#         ).fit(untreated_scaled)
#         _, indices = nn.kneighbors(treated_scaled)
#         matched_data = pd.concat([
#             data_clean[data_clean[treatment_col] == 1],
#             data_clean.loc[untreated_scaled.index[indices.flatten()]],
#         ])

#     for outcome in outcomes:
#         if matched_data[outcome].nunique() > 1:
#             model = sm.Logit(
#                 matched_data[outcome], sm.add_constant(matched_data[treatment_col])
#             ).fit(disp=0)
#             results[outcome] = {
#                 "pval": model.pvalues[treatment_col],
#                 "effect": model.params[treatment_col],
#             }
#             results[outcome] = {"pval": np.nan, "effect": np.nan}
#         else:
#             results[outcome] = {"pval": np.nan, "effect": np.nan}
#     return results


# treatment = "diploma"
# covariates = [
#     "age",
#     "income",
#     "n_child",
#     "type_area",
#     "smoking",
#     "alcohol",
#     "phys_active",
# ]

# final_tables = {}
# for g_name, g_df in [("Men", df_men), ("Women", df_women)]:
#     psm_res = perform_matching(g_df, treatment, targets, covariates, method="psm")
#     mah_res = perform_matching(
#         g_df, treatment, targets, covariates, method="mahalanobis"
#     )

#     combined = pd.DataFrame({
#         "Disease": targets,
#         "PSM Effect": [psm_res.get(t)["effect"] for t in targets],
#         "PSM P-Value": [psm_res.get(t)["pval"] for t in targets],
#         "Mahalanobis Effect": [mah_res.get(t)["effect"] for t in targets],
#         "Mahalanobis P-Value": [mah_res.get(t)["pval"] for t in targets],
#     })
#     combined["PSM Sig"] = combined["PSM P-Value"] < 0.05
#     combined["Mah Sig"] = combined["Mahalanobis P-Value"] < 0.05
#     final_tables[g_name] = combined

# for title, table in final_tables.items():
#     print(f"\n--- {title} ---")
#     display(table)


import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors


def perform_matching(
    data, treatment_col, outcomes, match_covars, outcome_covars, method="psm"
):
    # Drop NAs for all variables used in the pipeline
    all_vars = list(set(match_covars + outcome_covars + [treatment_col]))
    data_clean = data.dropna(subset=all_vars).copy()

    X_match = data_clean[match_covars]
    T = data_clean[treatment_col]
    results = {}

    # 1. Matching Phase (Strictly using match_covars)
    if method == "psm":
        lr = LogisticRegression(max_iter=1000)
        lr.fit(X_match, T)
        data_clean["ps"] = lr.predict_proba(X_match)[:, 1]

        treated = data_clean[data_clean[treatment_col] == 1]
        untreated = data_clean[data_clean[treatment_col] == 0]

        nn = NearestNeighbors(n_neighbors=1).fit(untreated[["ps"]])
        _, indices = nn.kneighbors(treated[["ps"]])

        matched_data = pd.concat([treated, untreated.iloc[indices.flatten()]])

    elif method == "mahalanobis":
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_match)
        X_scaled_df = pd.DataFrame(
            X_scaled, index=data_clean.index, columns=match_covars
        )

        treated_scaled = X_scaled_df[data_clean[treatment_col] == 1]
        untreated_scaled = X_scaled_df[data_clean[treatment_col] == 0]

        # Pseudo-inverse covariance matrix for Mahalanobis
        vi = np.linalg.pinv(np.cov(X_scaled.T))
        nn = NearestNeighbors(
            n_neighbors=1, metric="mahalanobis", metric_params={"VI": vi}
        ).fit(untreated_scaled)

        _, indices = nn.kneighbors(treated_scaled)
        matched_data = pd.concat([
            data_clean[data_clean[treatment_col] == 1],
            data_clean.loc[untreated_scaled.index[indices.flatten()]],
        ])

    # 2. Post-Matching Regression & ATE Calculation
    for outcome in outcomes:
        if matched_data[outcome].nunique() > 1:
            # Build design matrix with ALL control variables + treatment
            X_outcome = matched_data[outcome_covars]
            X_outcome = sm.add_constant(X_outcome)
            X_outcome[treatment_col] = matched_data[treatment_col]

            try:
                model = sm.Logit(matched_data[outcome], X_outcome).fit(disp=0)

                # Calculate Risk Difference (ATE) based on predicted probabilities
                X_treated = X_outcome.copy()
                X_treated[treatment_col] = 1

                X_untreated = X_outcome.copy()
                X_untreated[treatment_col] = 0

                prob_treated = model.predict(X_treated)
                prob_untreated = model.predict(X_untreated)

                ate_risk_difference = (prob_treated - prob_untreated).mean()

                results[outcome] = {
                    "pval": model.pvalues[treatment_col],
                    "effect": ate_risk_difference,  # Correct marginal effect
                }
            except Exception as e:
                # Catch perfectly separated data or convergence issues
                results[outcome] = {"pval": np.nan, "effect": np.nan}
        else:
            results[outcome] = {"pval": np.nan, "effect": np.nan}

    return results


# Implementation logic
treatment = "diploma"

# Variables strictly for matching (pre-treatment)
matching_covariates = ["age", "type_area", "invalid"]

# Variables for the post-matching logistic regression
outcome_covariates = [
    "age",
    "income",
    "n_child",
    "type_area",
    "invalid",
    "mar_st",
    "visit_doctor",
    "work",
    "alcohol",
    "smoking",
    "phys_active",
]

final_tables = {}
for g_name, g_df in [("Men", df_men), ("Women", df_women)]:
    psm_res = perform_matching(
        g_df, treatment, targets, matching_covariates, outcome_covariates, method="psm"
    )
    mah_res = perform_matching(
        g_df,
        treatment,
        targets,
        matching_covariates,
        outcome_covariates,
        method="mahalanobis",
    )

    combined = pd.DataFrame({
        "Disease": targets,
        "PSM Effect": [psm_res.get(t, {}).get("effect", np.nan) for t in targets],
        "PSM P-Value": [psm_res.get(t, {}).get("pval", np.nan) for t in targets],
        "Mahalanobis Effect": [
            mah_res.get(t, {}).get("effect", np.nan) for t in targets
        ],
        "Mahalanobis P-Value": [
            mah_res.get(t, {}).get("pval", np.nan) for t in targets
        ],
    })
    combined["PSM Sig"] = combined["PSM P-Value"] < 0.05
    combined["Mah Sig"] = combined["Mahalanobis P-Value"] < 0.05
    final_tables[g_name] = combined

c:\Users\alexv\Econometrics-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\alexv\Econometrics-project\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\alexv\Econometrics-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100


--- Men ---


,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,NaN,NaN,NaN,NaN,False,False
1,lungs,NaN,NaN,NaN,NaN,False,False
2,liver,NaN,NaN,NaN,NaN,False,False
3,kidneys,NaN,NaN,NaN,NaN,False,False
4,stomach,NaN,NaN,NaN,NaN,False,False
5,spine,NaN,NaN,NaN,NaN,False,False
6,diabetes,NaN,NaN,NaN,NaN,False,False
7,hypertension,NaN,NaN,NaN,NaN,False,False
8,joints,NaN,NaN,NaN,NaN,False,False
9,ENT_organs,NaN,NaN,NaN,NaN,False,False



--- Women ---


,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,NaN,NaN,NaN,NaN,False,False
1,lungs,NaN,NaN,NaN,NaN,False,False
2,liver,NaN,NaN,NaN,NaN,False,False
3,kidneys,NaN,NaN,NaN,NaN,False,False
4,stomach,NaN,NaN,NaN,NaN,False,False
5,spine,NaN,NaN,NaN,NaN,False,False
6,diabetes,NaN,NaN,NaN,NaN,False,False
7,hypertension,NaN,NaN,NaN,NaN,False,False
8,joints,NaN,NaN,NaN,NaN,False,False
9,ENT_organs,NaN,NaN,NaN,NaN,False,False


### Анализ результатов

Выше представлена часть из 60 моделей (15 заболеваний × 2 пола × 2 метода).

1. **Propensity Score Matching (PSM)**: Позволяет сбалансировать группы образованных и необразованных по ковариатам (возраст, доход и т.д.) на основе вероятности получения воздействия.
2. **Mahalanobis Distance**: Проводит поиск «близнецов» на основе многомерного расстояния.

Если `P-Value < 0.05`, это означает, что после выравнивания групп выбранный фактор сохраняет статистически значимую связь с конкретным заболеванием у данного пола.